In [68]:

import mediapipe as mp
import cv2
import os

In [69]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = "hand_landmarker.task"


In [70]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode



In [71]:
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.3
)




In [72]:
# import time
# cap = cv2.VideoCapture(1)
# time.sleep(2)
#
#
# with HandLandmarker.create_from_options(options) as landmarker:
#     cap = cv2.VideoCapture(1)
#
#     if cap.isOpened() is False:
#         print("Error opening video stream or file")
#
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             print("Could not read frame")
#             break
#
#         h, w = frame.shape[:2]
#         rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
#         result = landmarker.detect(mp_image)
#         if result.hand_landmarks:
#             for hand_landmark in result.hand_landmarks:
#                 for lm in hand_landmark:
#                     x = int(lm.x*w)
#                     y = int(lm.y*h)
#
#                     cv2.circle(frame, (x,y), 3, (0,255,), -1)
#
#
#         cv2.imshow("Frame", frame)
#         if cv2.waitKey(1) == ord('q'):
#             break
#
#     cap.release()
#     cv2.destroyAllWindows()
#





In [ ]:
import pandas as pd

handPositions = []
PATH = "/Volumes/T7/videos/Surdobot_VideoBase/07_11/resized_cropped_09_12_1_01_02-1-of-20.mp4"
out_path = "/Volumes/T7/videos/test"
file_name = "resized_cropped_09_12_1_01_02-1-of-20.mp4"
frame_idx = 0
with HandLandmarker.create_from_options(options) as landmarker:
    cap = cv2.VideoCapture(PATH)
    if not cap.isOpened():
        print("Could not open .mp4")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Total frames in video: {total_frames}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Could not read frame")
            break

        if frame_idx %3 == 0:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            res = landmarker.detect(mp_image)
            if res.hand_landmarks:
                h, w = frame.shape[:2]
                for hand_idx, hand_landmarks in enumerate(res.hand_landmarks):
                    wrist = hand_landmarks[0]
                    hand_label = res.handedness[hand_idx][0].display_name

                    row = {
                        "filename": file_name,
                        "frame": frame_idx,
                        "hand_label": hand_label,
                        "wrist_x": wrist.x,
                        "wrist_y": wrist.y
                    }

                    for j, lm in enumerate(hand_landmarks):
                        row[f"x{j}"] = lm.x - wrist.x
                        row[f"y{j}"] = lm.y - wrist.y

                         # draw each landmark
                        cx, cy = int(lm.x * w), int(lm.y * h)
                        cv2.circle(frame, (cx, cy), 2, (0, 255, 0), -1)

                    # draw wrist label
                    wx, wy = int(wrist.x * w), int(wrist.y * h)
                    cv2.putText(frame, hand_label, (wx, wy - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                    handPositions.append(row)

        frame_idx += 1
        cv2.imwrite(f"{out_path}/frame_{frame_idx}.jpg", frame)

df = pd.DataFrame(handPositions)









I0000 00:00:1781005544.927043 1226826 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1781005544.930531 1226829 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781005544.939835 1226829 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Total frames in video: 652


In [ ]:
df

In [ ]:
outpath = "/Volumes/T7/videos"
file_name = "frame_24.png"
cap = cv2.VideoCapture(PATH)
frame_id = 0
if not cap.isOpened():
    print("Could not open .mp4")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Could not read frame")
        break

    h, w = frame.shape[:2]
    if frame_id == 24:

        for i in range(0, 21):
            actual_x = df["wrist_x"].iloc[0] + df[f"x{i}"].iloc[0]
            actual_y = df["wrist_y"].iloc[0] + df[f"y{i}"].iloc[0]

            x, y = int(actual_x*w), int(actual_y*h)
            cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

        cv2.imwrite(os.path.join(outpath, file_name), frame)
        break
    if cv2.waitKey(1) == ord('q'):
        break
    frame_id += 1

cap.release()
cv2.destroyAllWindows()



In [ ]:
df.columns.tolist

In [ ]:
print(df["frame"].iloc[0])      # what frame is this row from?
print(df["filename"].iloc[0])   # what video is this row from?
print(df["hand_label"].iloc[0]) # Left or Right?
print(df["wrist_x"].iloc[0], df["wrist_y"].iloc[0])  # where is the wrist?

In [ ]:
# Check frames where 2 hands were detected
two_hand_frames = df.groupby("frame").filter(lambda x: len(x) > 1)
print(two_hand_frames[["frame", "hand_label", "wrist_x", "wrist_y"]].head(10))

# Count label distribution

In [ ]:
cap = cv2.VideoCapture(PATH)
frame_index = 0
if not cap.isOpened():
    print("Could not open .mp4")

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        print("Could not read frame")
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)




